# 🎵 Music Search with CLAP + Weaviate

Search for music using natural language:
- "chill lo-fi beats for studying"
- "aggressive metal with fast drums"
- "90s hip hop with jazz samples"

## How it works

1. **92 audio tracks** are already embedded and stored in Weaviate
2. **CLAP** embeds your text query into the same vector space as the audio
3. **Weaviate** finds the nearest audio embeddings
4. You get matching songs!

This is **true cross-modal search**: text in, audio out!

---

## Step 1: Setup Environment

First, let's enable GPU and install the required packages.

**Important:** Go to `Runtime → Change runtime type → T4 GPU` before running this notebook!

In [ ]:
# Install required packages
!pip install -q transformers torch weaviate-client gradio

print("✅ Packages installed!")

In [ ]:
# Check if GPU is available
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print("   (CPU will work but be slower)")

## Step 2: Set Up Weaviate Cloud

Get your free Weaviate sandbox:

1. Go to https://console.weaviate.cloud/
2. Sign up / Log in
3. Click **Create Cluster** → Choose **Sandbox** (free)
4. Wait for it to be ready (~1 min)
5. Click on your cluster → **API Keys** → Copy the key
6. Copy the **Cluster URL** (looks like `https://xxx.weaviate.network`)

In [ ]:
# ⬇️ PASTE YOUR WEAVIATE CREDENTIALS HERE ⬇️

WEAVIATE_URL = ""  # e.g., "https://my-sandbox-abc123.weaviate.network"
WEAVIATE_API_KEY = ""  # e.g., "your-api-key-here"

# ⬆️ PASTE YOUR WEAVIATE CREDENTIALS HERE ⬆️

if WEAVIATE_URL and WEAVIATE_API_KEY:
    print("✅ Credentials set!")
else:
    print("❌ Please paste your Weaviate URL and API key above")

## Step 3: Load the CLAP Model

CLAP (from LAION) is like CLIP but for audio:
- **CLIP**: Text ↔ Image (OpenAI)
- **CLAP**: Text ↔ Audio (LAION, open source, FREE)

Both text and audio get embedded into the same 512-dimensional vector space.

In [ ]:
from transformers import ClapModel, ClapProcessor

print("Loading CLAP model (first run downloads ~600MB)...")

model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
model = model.to(device)
model.eval()

print("✅ CLAP model loaded!")

## Step 4: Connect to Weaviate

The audio embeddings are already uploaded! Just connect and search.

In [ ]:
import weaviate
from weaviate.classes.init import Auth

# Connect to Weaviate Cloud
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY)
)

print(f"✅ Connected to Weaviate!")
print(f"   Ready: {client.is_ready()}")

# Check the collection
COLLECTION_NAME = "MusicTrack"
collection = client.collections.get(COLLECTION_NAME)
count = collection.aggregate.over_all(total_count=True).total_count
print(f"   Tracks in database: {count}")

## Step 5: Search UI 🎵

Launch an interactive search interface!

In [ ]:
import gradio as gr
import numpy as np

def search_music(query: str, num_results: int = 5):
    """Search for music and return formatted results."""
    if not query.strip():
        return "Please enter a search query."

    # Embed the query with CLAP
    inputs = processor(text=[query], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.get_text_features(**inputs)

    # Extract embedding tensor
    if hasattr(outputs, 'text_embeds'):
        query_embedding = outputs.text_embeds
    elif hasattr(outputs, 'pooler_output'):
        query_embedding = outputs.pooler_output
    else:
        query_embedding = outputs

    # Convert to 1D and normalize
    query_embedding = query_embedding.cpu().numpy().flatten()
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # Search Weaviate
    collection = client.collections.get(COLLECTION_NAME)
    results = collection.query.near_vector(
        near_vector=query_embedding.tolist(),
        limit=num_results,
        return_metadata=["distance"]
    )

    # Format results as HTML
    html = f"<h3>🔍 Results for: \"{query}\"</h3>"

    if not results.objects:
        return html + "<p>No results found.</p>"

    for i, obj in enumerate(results.objects, 1):
        props = obj.properties
        similarity = 1 - (obj.metadata.distance or 0)
        youtube_url = props.get('youtube_url', '')

        html += f"""
        <div style="border: 1px solid #ddd; border-radius: 8px; padding: 12px; margin: 8px 0; background: #f9f9f9;">
            <strong>#{i}</strong> &nbsp; Similarity: <span style="color: green;">{similarity:.2%}</span>
            <p style="margin: 8px 0;">📝 {props.get('caption', 'No caption')}</p>
            <a href="{youtube_url}" target="_blank" style="color: #1a73e8;">🎬 Listen on YouTube</a>
        </div>
        """

    return html

# Create Gradio interface
demo = gr.Interface(
    fn=search_music,
    inputs=[
        gr.Textbox(
            label="Describe the music you're looking for",
            placeholder="e.g., chill lo-fi beats for studying",
            lines=2
        ),
        gr.Slider(minimum=1, maximum=10, value=5, step=1, label="Number of results")
    ],
    outputs=gr.HTML(label="Results"),
    title="🎵 Music Search",
    description="Search for music using natural language. Powered by CLAP + Weaviate.",
    examples=[
        ["chill lo-fi beats for studying", 5],
        ["aggressive metal with fast drums", 5],
        ["upbeat electronic dance music", 5],
        ["sad piano ballad", 5],
        ["jazz with saxophone", 5],
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

## Step 6: Cleanup (Optional)

Close the Weaviate connection when done.

In [ ]:
# Close connection when done
client.close()
print("✅ Weaviate connection closed")

---

## 🎉 Congratulations!

You've built a **true cross-modal** music search engine!

### What makes this special

- **Audio embeddings**: The database contains embeddings from actual audio waveforms, not text
- **Cross-modal search**: Your text query finds matching audio through shared embedding space
- **CLAP magic**: Text and audio live in the same 512-dimensional vector space

### Tech Stack

1. **CLAP** - Embeds text and audio in the same space (LAION, FREE)
2. **Weaviate** - Vector database for fast similarity search
3. **Gradio** - Interactive web UI

### Key Concepts

- **Embeddings**: Convert text/audio into numerical vectors that capture meaning
- **Vector similarity**: Similar meanings → vectors close together
- **Cross-modal search**: Search audio using text (or vice versa)

### Resources

- [CLAP on Hugging Face](https://huggingface.co/laion/clap-htsat-unfused)
- [MusicCaps Dataset](https://huggingface.co/datasets/google/MusicCaps)
- [Weaviate Docs](https://weaviate.io/developers/weaviate)